loading data, quality checks, table schema decisions filters

In [41]:
#import library
import pandas as pd

In [ ]:
#load data
aisles=pd.read_csv("C:/Users/tcham/Desktop/ml-exam/data/aisles.csv")
departments=pd.read_csv("C:/Users/tcham/Desktop/ml-exam/data/departments.csv")
orders=pd.read_csv("C:/Users/tcham/Desktop/ml-exam/data/orders.csv")
products=pd.read_csv("C:/Users/tcham/Desktop/ml-exam/data/products.csv")
order_products__prior=pd.read_csv("C:/Users/tcham/Desktop/ml-exam/data/order_products__prior.csv")
order_products__train=pd.read_csv("C:/Users/tcham/Desktop/ml-exam/data/order_products__train.csv")

In [50]:
def inventory(df: pd.DataFrame, name: str, n: int = 10) -> None:
    print("\n" + "="*100)
    print(f"TABLE: {name}")
    print(f"Shape (rows, cols): {df.shape[0]:,} rows x {df.shape[1]} cols")
    print("\nColumns:")
    print(df.columns.tolist())
    print(f"\nHead({n}):")
    display(df.head(n))

In [51]:
inventory(aisles, "aisles")
inventory(departments, "departments")
inventory(products, "products")
inventory(orders, "orders")
inventory(order_products__prior, "order_products__prior")
inventory(order_products__train, "order_products__train")



TABLE: aisles
Shape (rows, cols): 134 rows x 2 cols

Columns:
['aisle_id', 'aisle']

Head(10):


,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation
5,6,other
6,7,packaged meat
7,8,bakery desserts
8,9,pasta sauce
9,10,kitchen supplies



TABLE: departments
Shape (rows, cols): 21 rows x 2 cols

Columns:
['department_id', 'department']

Head(10):


,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol
5,6,international
6,7,beverages
7,8,pets
8,9,dry goods pasta
9,10,bulk



TABLE: products
Shape (rows, cols): 49,688 rows x 4 cols

Columns:
['product_id', 'product_name', 'aisle_id', 'department_id']

Head(10):


...

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0
5,3367565,1,prior,6,2,7,19.0
6,550135,1,prior,7,1,9,20.0
7,3108588,1,prior,8,1,14,14.0
8,2295261,1,prior,9,1,16,0.0
9,2550362,1,prior,10,4,8,30.0



TABLE: order_products__prior
Shape (rows, cols): 32,434,489 rows x 4 cols

Columns:
['order_id', 'product_id', 'add_to_cart_order', 'reordered']

Head(10):


,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0
5,2,17794,6,1
6,2,40141,7,1
7,2,1819,8,1
8,2,43668,9,0
9,3,33754,1,1



TABLE: order_products__train
Shape (rows, cols): 1,384,617 rows x 4 cols

Columns:
['order_id', 'product_id', 'add_to_cart_order', 'reordered']

Head(10):


,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0
3,1,49683,4,0
4,1,43633,5,1
5,1,13176,6,0
6,1,47209,7,0
7,1,22035,8,1
8,36,39612,1,0
9,36,19660,2,1


In [43]:
# 1-check number of order in prior, train and text
print("distribution of eval_set:")
print(orders["eval_set"].value_counts())

distribution of eval_set:
eval_set
prior    3214874
train     131209
test       75000
Name: count, dtype: int64


In [44]:
#2- Check there is no products 2 time in the same order to avoid
a= order_products__prior.duplicated(["order_id","product_id"]).sum()
b= order_products__train.duplicated(["order_id","product_id"]).sum()
print(a,b)

0 0


In [45]:
# 3- Ckeck that all the products in order_products exist in products
missing_prod_prior = order_products__prior.loc[~order_products__prior["product_id"].isin(products["product_id"]), "product_id"].nunique()
missing_prod_train = order_products__train.loc[~order_products__train["product_id"].isin(products["product_id"]), "product_id"].nunique()

print("count of missing product_id in products:")
print("prior:", missing_prod_prior)
print("train:", missing_prod_train)


count of missing product_id in products:
prior: 0
train: 0


In [52]:
# 4- check that order_products__train have all the order in orders(eval_set=train)

train_order_ids = set(orders.loc[orders["eval_set"] == "train", "order_id"].unique())
op_train = order_products__train[order_products__train["order_id"].isin(train_order_ids)].copy()

print("Nb train order_ids:", len(train_order_ids))
print("op_train rows kept:", op_train.shape[0])


Nb train order_ids: 131209
op_train rows kept: 1384617
